In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
# ======================================================
# 1. Installer les libs
# ======================================================
!pip install transformers datasets peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 18.8 MB/s eta 0:00:00


In [ ]:
# ======================================================
# 1. Imports
# ======================================================
import torch
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# ======================================================
# 2. Configs
# ======================================================
MODEL_NAME = "google/flan-t5-large"   # 🔹 Modèle Flan
DATA_FILE = "data.jsonl"
OUTPUT_DIR = "./flan-t5-class"
MAX_LENGTH = 1024
EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 8

# ======================================================
# 3. Charger modèle et tokenizer
# ======================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# ======================================================
# 4. Charger dataset
# ======================================================
df = pd.read_json(DATA_FILE, lines=True)
dataset = Dataset.from_pandas(df)

# ======================================================
# 5. Formatter les données
# ======================================================
def format_data_jsonl(ex):
    output_classes = []
    for cls in ex.get("output", {}).get("classes", []):
        output_classes.append({
            "text": cls.get("text", "N/A"),
            "class": cls.get("class", "N/A"),
            "attributes": [
                {"text": attr.get("text","N/A"), "name": attr.get("name","N/A"), "visibility": attr.get("visibility","N/A"), "type": attr.get("type","N/A")}
                for attr in cls.get("attributes", [])
            ],
            "methods": [
                {"text": m.get("text","N/A"), "name": m.get("name","N/A"), "visibility": m.get("visibility","N/A"),
                 "parameters": [{"name": p.get("name","N/A"), "type": p.get("type","N/A")} for p in m.get("parameters",[])],
                 "return_type": m.get("return_type","N/A")
                }
                for m in cls.get("methods", [])
            ]
        })

    output_relations = [
        {
            "text": r.get("text","N/A"),
            "from": r.get("from","N/A"),
            "to": r.get("to","N/A"),
            "type": r.get("type","N/A"),
            "name": r.get("name","N/A"),
            "multiplicityFrom": r.get("multiplicityFrom","N/A"),
            "multiplicityTo": r.get("multiplicityTo","N/A")
        }
        for r in ex.get("output", {}).get("relations", [])
    ]

    json_line = {
        "output": {
            "classes": output_classes,
            "relations": output_relations
        }
    }

    return {"text": json.dumps(json_line, ensure_ascii=False)}

formatted_dataset = dataset.map(format_data_jsonl)

# ======================================================
# 6. Tokenization
# ======================================================
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    # Pour Flan (Seq2Seq), on met la même cible = text (auto-encoding)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples['text'],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length"
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names
)

# ======================================================
# 7. Préparer LoRA
# ======================================================
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=["q", "v"],  # pour T5 / Flan
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# ======================================================
# 8. TrainingArguments
# ======================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
)

# ======================================================
# 9. Trainer avec DataCollator
# ======================================================
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=peft_model,
    padding=True
)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)

# ======================================================
# 10. Lancer fine-tuning
# ======================================================
trainer.train()

# ======================================================
# 11. Sauvegarde
# ======================================================
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modèle et tokenizer sauvegardés dans {OUTPUT_DIR}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

trainable params: 51,118,080 || all params: 2,665,459,968 || trainable%: 1.9178


It is strongly recommended to train Gemma2 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.084100
20,0.857100
30,0.802900
40,0.714400
50,0.742500
60,0.716000
70,0.675100
80,0.632000
90,0.687500
100,0.626700


Modèle et tokenizer sauvegardés dans ./gemma-2-2b-it-class
